# Fine-tune Qwen 2.5 7B Instruct on BrainCode, then run ProofWriter

**Part 1 (phase 1)** fine-tunes Qwen 2.5 7B Instruct with LoRA (Unsloth) on the validated BrainCode translations (`data/train.jsonl`) and saves the adapter to your Drive.

**Part 2 (phases 3 and 4a)** serves the base model and the adapter with vLLM, and runs the 70 ProofWriter items (depth-5 OWA test, proof depth 3 and 4):
- `baseline`: plain Qwen 2.5 on the English items;
- `braincode_ft`: the fine-tuned Qwen on the items' BrainCode (Gemini's successful translations, phase 2). It ends with its conclusion in BrainCode, which Gemini back-translators read back in the repository (phase 4b).

### Before you start
1. Run phase 2 locally first (`evaluations/proofwriter/pw.py translate`, then `package`). `benchmark/sample.jsonl` must hold the translations.
2. Upload this folder to **MyDrive** as `finetune-qwen.zip` (fast) or as the folder `MyDrive/finetune-qwen`.
3. **Runtime → Change runtime type → GPU.** An **A100** is best. An **L4** works. A **T4** works with the 4-bit model.

Run the cells top to bottom. Part 2 starts with a runtime restart.

## Part 1: fine-tune (phase 1)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
os.chdir('/content')     # never work inside the Drive mount: if it drops, every import fails (Errno 107)
from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/finetune-qwen'   # results go here on Drive (change if elsewhere)
ZIP = '/content/drive/MyDrive/finetune-qwen.zip'  # the folder as one zip: copies in seconds
WORK = '/content/finetune-qwen'                   # a local copy to work from: fast, and immune to Drive drops
if not os.path.exists(f'{WORK}/data/train.jsonl'):  # after a session restart the local copy is still there
    if os.path.exists(ZIP):
        !unzip -q -o "{ZIP}" -d /content
    else:                                          # file by file from Drive: slow (minutes), shows progress
        assert os.path.exists(f'{FOLDER}/data/train.jsonl'), 'upload finetune-qwen.zip (or the folder) to MyDrive'
        !rsync -a --info=progress2 --exclude runs --exclude outputs --exclude models "{FOLDER}/" "{WORK}/"
OUT = f'{FOLDER}/outputs'                          # adapter and zipped results are saved on Drive
os.makedirs(OUT, exist_ok=True)
os.makedirs(f'{FOLDER}/runs', exist_ok=True)       # benchmark runs are written straight to Drive (resumable)
if not os.path.islink(f'{WORK}/runs'):
    os.symlink(f'{FOLDER}/runs', f'{WORK}/runs')
%cd {WORK}

In [ ]:
%%capture
!pip install unsloth

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # less fragmentation (set before torch loads)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'   # plain downloads: the accelerated paths can truncate tokenizer.json
os.environ['HF_HUB_DISABLE_XET'] = '1'
import torch

GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
LOAD_IN_4BIT = GPU_GB < 40          # A100: 16-bit LoRA; L4/T4: 4-bit QLoRA
BASE_MODEL = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' if LOAD_IN_4BIT else 'unsloth/Qwen2.5-7B-Instruct'
MAX_SEQ_LEN = 8192 if GPU_GB >= 40 else 4096
READ_DIRECTION = True   # also train BrainCode -> English (each translation read back). Phase 4 gives the model
                        # BrainCode to read; False trains only English -> BrainCode, as in the Llama run.
EPOCHS = 2
LEARNING_RATE = 2e-4
LORA_R = 16
BATCH, GRAD_ACC = (2, 4) if GPU_GB >= 70 else (1, 8)
ADAPTER_DIR = f'{OUT}/braincode-qwen-lora'
print(f'GPU {GPU_GB:.0f} GB -> {BASE_MODEL}, max length {MAX_SEQ_LEN}, batch {BATCH} x {GRAD_ACC}')

### Training data
Each translation gives a **write** example: the task in English, answered with its BrainCode. With `READ_DIRECTION`, it also gives a **read** example: the BrainCode, answered with the task in English. The loss is on the answers only.

In [ ]:
import json, random

WRITE_SYSTEM = 'You translate tasks into BrainCode, a formal language with a fixed glossary of symbols. You do not solve the task.'
READ_SYSTEM = 'You read and write BrainCode, a formal language with a fixed glossary of symbols.'

def to_chats(row):
    fenced = '```braincode\n' + row['braincode'] + '\n```'
    chats = [[{'role': 'system', 'content': WRITE_SYSTEM},
              {'role': 'user', 'content': 'Translate this task into BrainCode:\n\n' + row['item']},
              {'role': 'assistant', 'content': fenced}]]
    if READ_DIRECTION:
        chats.append([{'role': 'system', 'content': READ_SYSTEM},
                      {'role': 'user', 'content': 'Write this BrainCode task in plain English:\n\n' + fenced},
                      {'role': 'assistant', 'content': row['item']}])
    return chats

def load(name):
    with open(f'{WORK}/data/{name}.jsonl', encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]

train_rows, val_rows = load('train'), load('val')
print(json.load(open(f'{WORK}/data/manifest.json'))['by_dataset'])
print(len(train_rows), 'train /', len(val_rows), 'validation translations;',
      'write + read examples' if READ_DIRECTION else 'write examples only')

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

def load_base():
    return FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=LOAD_IN_4BIT, dtype=None)

free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb > (8 if LOAD_IN_4BIT else 20), (
    f'only {free_gb:.1f} GB of GPU memory is free: an earlier model is still loaded. '
    'Runtime -> Restart session, then run the cells again from the Drive-mount cell (skip pip install).')
try:
    model, tokenizer = load_base()
except Exception as e:
    if 'expected value' not in str(e) and 'tokenizer' not in str(e).lower():
        raise
    # a truncated download ("expected value at line 1 column 1" from tokenizer.json): clear it and retry once
    import glob, shutil
    print('load failed:', e, '\n-> clearing the cached download and retrying once')
    for d in glob.glob(os.path.expanduser('~/.cache/huggingface/hub/models--' + BASE_MODEL.replace('/', '--') + '*')):
        shutil.rmtree(d, ignore_errors=True)
    model, tokenizer = load_base()
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=2026)
tokenizer = get_chat_template(tokenizer, chat_template='qwen-2.5')

In [ ]:
from datasets import Dataset

def as_text(rows):
    texts = [tokenizer.apply_chat_template(c, tokenize=False) for r in rows for c in to_chats(r)]
    lengths = [len(tokenizer(t).input_ids) for t in texts]
    kept = [t for t, n in zip(texts, lengths) if n <= MAX_SEQ_LEN]
    print(f'{len(kept)}/{len(texts)} examples fit in {MAX_SEQ_LEN} tokens (longest {max(lengths)})')
    random.Random(2026).shuffle(kept)
    return Dataset.from_dict({'text': kept})

train_ds, val_ds = as_text(train_rows), as_text(val_rows)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC,
        num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, lr_scheduler_type='cosine', warmup_steps=10,
        weight_decay=0.01, optim='adamw_8bit', logging_steps=5, eval_strategy='epoch', save_strategy='no',
        bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
        output_dir='/content/checkpoints', report_to='none', seed=2026))
# learn only the answers, not the prompts
trainer = train_on_responses_only(trainer,
    instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
stats = trainer.train()
print(stats)
print(trainer.evaluate())

In [ ]:
# Save the LoRA adapter to Drive (~160 MB). Part 2 serves it with vLLM.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('saved', ADAPTER_DIR)

# Optional: push the adapter to the Hugging Face Hub (Colab secret HF_TOKEN, write permission)
PUSH_TO_HUB = False
HUB_REPO = 'your-username/braincode-qwen2.5-7b-lora'
if PUSH_TO_HUB:
    from google.colab import userdata
    model.push_to_hub(HUB_REPO, token=userdata.get('HF_TOKEN'), private=True)
    tokenizer.push_to_hub(HUB_REPO, token=userdata.get('HF_TOKEN'), private=True)

### Sanity check: translate a held-out item and run the BrainCode checker on it

In [ ]:
import sys
sys.path.insert(0, f'{WORK}/braincode_kit')
!pip -q install rank_bm25
from rag.retrieve import Retriever, render_check

FastLanguageModel.for_inference(model)
row = val_rows[0]
msgs = to_chats(row)[0][:2]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=2048, temperature=0.7, top_p=0.8, do_sample=True)
answer = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print(answer[:3000])

checker = Retriever(dense=False, glossary_path=f'{WORK}/braincode_kit/reference/glossary.jsonl',
                    index_dir=f'{WORK}/braincode_kit/rag/index', use_llm=False)
print(render_check(checker.check(answer, [])))

## Part 2: serve the models and run ProofWriter (phases 3 and 4a)

**First: Runtime → Restart session.** This frees the GPU memory from training and gives vLLM a clean environment. Files on Drive and in `/content` stay. Python variables are lost, so the next cell sets the paths again.

If your session disconnected after Part 1, start here: the adapter is on Drive.

In [ ]:
import os
os.chdir('/content')     # never work inside the Drive mount: if it drops, every import fails (Errno 107)
from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/finetune-qwen'   # results go here on Drive (change if elsewhere)
ZIP = '/content/drive/MyDrive/finetune-qwen.zip'  # the folder as one zip: copies in seconds
WORK = '/content/finetune-qwen'                   # a local copy to work from: fast, and immune to Drive drops
if not os.path.exists(f'{WORK}/data/train.jsonl'):  # after a session restart the local copy is still there
    if os.path.exists(ZIP):
        !unzip -q -o "{ZIP}" -d /content
    else:                                          # file by file from Drive: slow (minutes), shows progress
        assert os.path.exists(f'{FOLDER}/data/train.jsonl'), 'upload finetune-qwen.zip (or the folder) to MyDrive'
        !rsync -a --info=progress2 --exclude runs --exclude outputs --exclude models "{FOLDER}/" "{WORK}/"
OUT = f'{FOLDER}/outputs'                          # adapter and zipped results are saved on Drive
os.makedirs(OUT, exist_ok=True)
os.makedirs(f'{FOLDER}/runs', exist_ok=True)       # benchmark runs are written straight to Drive (resumable)
if not os.path.islink(f'{WORK}/runs'):
    os.symlink(f'{FOLDER}/runs', f'{WORK}/runs')
%cd {WORK}

import torch, json, shutil
ADAPTER_DIR = f'{FOLDER}/outputs/braincode-qwen-lora'
assert os.path.exists(f'{ADAPTER_DIR}/adapter_config.json'), 'no adapter: run Part 1 first'
if os.path.exists(f'{FOLDER}/benchmark/sample.jsonl'):   # a newer benchmark/ uploaded to Drive (after phase 2) wins
    shutil.copytree(f'{FOLDER}/benchmark', f'{WORK}/benchmark', dirs_exist_ok=True)
if os.path.exists(f'{FOLDER}/run_proofwriter.py'):       # so does a newer runner
    shutil.copy2(f'{FOLDER}/run_proofwriter.py', f'{WORK}/run_proofwriter.py')
items = [json.loads(l) for l in open(f'{WORK}/benchmark/sample.jsonl', encoding='utf-8')]
print(len(items), 'items,', sum(bool(i['braincode']) for i in items), 'with a successful BrainCode translation')
if not any(i['braincode'] for i in items):
    print('No translations in benchmark/ yet: the baseline can run now. For phase 4, run `pw.py package` locally, '
          f'upload finetune-qwen/benchmark/ to {FOLDER}/benchmark, and rerun this cell.')
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
if GPU_GB >= 20:      # A100 / L4: 16-bit weights
    SERVE_MODEL, EXTRA, MAX_MODEL_LEN = 'Qwen/Qwen2.5-7B-Instruct', '', 32768 if GPU_GB >= 40 else 20480
else:                 # T4: 16-bit weights do not fit; serve the 4-bit model (fp16, no bf16 on T4)
    SERVE_MODEL, EXTRA, MAX_MODEL_LEN = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit', '--quantization bitsandbytes --dtype half', 16384
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
print(f'GPU {GPU_GB:.0f} GB -> serving {SERVE_MODEL} with {MAX_MODEL_LEN} tokens of context')

In [ ]:
%%capture
!pip install vllm

In [ ]:
# vLLM serves the base model as `base-qwen` and the fine-tuned adapter on top of it as `braincode-qwen`.
import subprocess, time, urllib.request
cmd = (f'vllm serve {SERVE_MODEL} --served-model-name base-qwen {EXTRA} '
       f'--enable-lora --lora-modules braincode-qwen={ADAPTER_DIR} --max-lora-rank 16 '
       f'--max-model-len {MAX_MODEL_LEN} --gpu-memory-utilization 0.90 --port 8000')
server = subprocess.Popen(cmd, shell=True, stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)

for _ in range(180):                       # model download + load: usually 3-8 minutes
    try:
        models = json.loads(urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5).read())
        print('server up:', [m['id'] for m in models['data']])
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError('vLLM exited; see /content/vllm.log:\n' + open('/content/vllm.log').read()[-3000:])
        time.sleep(10)
else:
    print(open('/content/vllm.log').read()[-3000:])

### Smoke test: 2 items, 1 run per condition (read the responses before the full run)

In [ ]:
!python run_proofwriter.py --model base-qwen --condition smoke_baseline --input english --runs 1 --limit 2
!python run_proofwriter.py --model braincode-qwen --condition smoke_braincode_ft --input braincode --runs 1 --limit 2
import glob
for run in sorted(glob.glob(f'{WORK}/runs/smoke_braincode_ft/*/r1')):
    print('=' * 30, run.split('/')[-2])
    print('--- reasoning (call 1) ---\n' + open(f'{run}/reasoning.md').read()[-3000:])
    print('--- verdict (call 2) ---\n' + open(f'{run}/response.md').read())

Check before the full run: the reasoning should derive facts with rules (not restate the problem), and the verdict should be a short block with the statement as `q` and one of the three verdict forms.

### Phase 3: the baseline, plain Qwen 2.5 on the English items (70 × 3)
Resumable: if Colab disconnects, rerun the cell. Finished runs are kept and errored runs are redone.

In [ ]:
!python run_proofwriter.py --model base-qwen --condition baseline --input english --runs 3

### Phase 4a: the fine-tuned Qwen on the successful BrainCode translations (× 3)
It ends each answer with its conclusion as a BrainCode block. Back in the repository, Gemini back-translators read that block (phase 4b).

In [ ]:
!python run_proofwriter.py --model braincode-qwen --condition braincode_ft --input braincode --runs 3

### Optional controls
- `braincode_base`: the plain model on the same BrainCode. Is any gain from fine-tuning, or from the representation alone?
- `baseline_ft`: the fine-tuned model on English. Did fine-tuning change plain reasoning?

In [ ]:
!python run_proofwriter.py --model base-qwen --condition braincode_base --input braincode --runs 3
!python run_proofwriter.py --model braincode-qwen --condition baseline_ft --input english --runs 3

### Progress, and a zip of all runs to bring back to the repository

In [ ]:
import glob, shutil
from collections import Counter
c = Counter()
for p in glob.glob(f'{FOLDER}/runs/*/*/r*/result.json'):
    r = json.load(open(p))
    c[(r['condition'], 'error' if r['status'] != 'ok' else 'awaiting back-translation' if r['correct'] is None
       else 'correct' if r['correct'] else 'wrong')] += 1
for (cond, state), n in sorted(c.items()):
    print(f'{cond:22} {state:26} {n}')
shutil.make_archive(f'{FOLDER}/outputs/runs', 'zip', f'{FOLDER}/runs')
print('zipped ->', f'{FOLDER}/outputs/runs.zip')